# Chapter 07 — Sequence Models

This notebook studies recurrent neural networks and how they process sequential data.

## Learning objectives
- Understand sequence inputs and recurrent hidden states.
- Implement a vanilla RNN for sequence prediction.
- Understand vanishing and exploding gradients.
- Explain the roles of LSTM gates and the cell state.
- Compare LSTM and GRU architectures.
- Build a small recurrent sentiment-classification demonstration.

**Scope note:** Tokenization, vocabulary design, padding strategies, embedding spaces, language modelling, and transformers will be studied systematically in **Chapter 10 — Embeddings & Language Models**. The small text-classification example here is only an application of recurrent models, not a full NLP unit.


## 1. Sequence data and sliding windows

For next-step prediction, a continuous series can be converted into input windows and their next values. With a window length of 5, `[1, 2, 3, 4, 5]` maps to target `6`.

With `batch_first=True`, recurrent input has shape `(batch, sequence_length, features)`.


In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "xpu" if hasattr(torch, "xpu") and torch.xpu.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("Device:", device)

values = np.arange(1, 1201, dtype=np.float32)
normalized = values / float(values.max())

sequence_length = 5
X, y = [], []
for i in range(len(normalized) - sequence_length):
    X.append(normalized[i:i + sequence_length])
    y.append(normalized[i + sequence_length])

X = torch.tensor(np.array(X), dtype=torch.float32).unsqueeze(-1)
y = torch.tensor(np.array(y), dtype=torch.float32).unsqueeze(-1)
print("X:", X.shape)
print("y:", y.shape)


Device: xpu
X: torch.Size([1195, 5, 1])
y: torch.Size([1195, 1])


## 2. Vanilla RNN for next-step prediction

At each time step, a vanilla RNN combines the current input with the previous hidden state to produce a new hidden state. The final time-step representation is passed to a linear layer.


In [2]:
class SequencePredictor(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.rnn = nn.RNN(1, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, hidden = self.rnn(x)
        return self.fc(output[:, -1, :])

model = SequencePredictor().to(device)
print("Output shape:", model(X[:4].to(device)).shape)


Output shape: torch.Size([4, 1])


## 3. Training and evaluation split

This randomized split is reproducible, but adjacent sliding windows overlap heavily. It is a teaching demonstration, not rigorous forecasting evaluation. For real forecasting, use a chronological split to prevent future observations leaking into training.


In [3]:
generator = torch.Generator().manual_seed(42)
indices = torch.randperm(len(X), generator=generator)
split = int(0.8 * len(indices))
train_idx, val_idx = indices[:split], indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

model = SequencePredictor().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    model.train()
    train_total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_total += loss.item() * len(xb)

    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            val_total += criterion(model(xb), yb).item() * len(xb)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | train {train_total/len(X_train):.6f} | val {val_total/len(X_val):.6f}")


Epoch  20 | train 0.000464 | val 0.000433
Epoch  40 | train 0.000239 | val 0.000236
Epoch  60 | train 0.000116 | val 0.000107
Epoch  80 | train 0.000055 | val 0.000049
Epoch 100 | train 0.000030 | val 0.000027


## 4. Vanishing and exploding gradients

Backpropagation through time (BPTT) propagates gradients across sequence steps.

- **Vanishing gradients:** gradient signals can become very small as they travel backwards, making it difficult to learn the influence of earlier inputs.
- **Exploding gradients:** gradient magnitudes can become very large, causing unstable parameter updates and potentially sharp loss spikes.

A loss spike alone does not prove exploding gradients. Inspect gradient norms. Gradient clipping is a common mitigation.


In [4]:
xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
model.train()
optimizer.zero_grad()
loss = criterion(model(xb), yb)
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float("inf"))
print("Global gradient norm:", float(grad_norm))
optimizer.zero_grad()


Global gradient norm: 0.002089613350108266


## 5. LSTM architecture

An LSTM maintains a **cell state** and a **hidden state**.

- **Forget gate:** controls what to retain from the previous cell state.
- **Input gate:** controls what candidate information to add.
- **Output gate:** controls what information to expose as the hidden state.

The separate cell state and gate structure can help LSTMs learn longer-range dependencies than a vanilla RNN.


In [5]:
lstm = nn.LSTM(input_size=1, hidden_size=16, batch_first=True).to(device)
sample_sequence = torch.randn(8, 5, 1, device=device)
output, (hidden, cell) = lstm(sample_sequence)
print("output:", output.shape)
print("hidden:", hidden.shape)
print("cell:", cell.shape)


output: torch.Size([8, 5, 16])
hidden: torch.Size([1, 8, 16])
cell: torch.Size([1, 8, 16])


## 6. GRU architecture

A GRU uses two gates:
- **Update gate:** controls how much previous hidden state is carried forward versus replaced.
- **Reset gate:** controls how much past information contributes to the candidate state.

Unlike an LSTM, a GRU does not maintain a separate cell state.


In [6]:
gru = nn.GRU(input_size=1, hidden_size=16, batch_first=True).to(device)
gru_output, gru_hidden = gru(sample_sequence)
print("output:", gru_output.shape)
print("hidden:", gru_hidden.shape)


output: torch.Size([8, 5, 16])
hidden: torch.Size([1, 8, 16])


## 7. LSTM vs GRU parameter counts

For a single-layer, unidirectional PyTorch recurrent layer with input size `I` and hidden size `H`, including both bias vectors:

- LSTM: `4 * (H*I + H*H + 2*H)`
- GRU: `3 * (H*I + H*H + 2*H)`

Parameter count does not directly determine runtime. Sequence length, batch size, operations, memory traffic, hardware, and kernel implementation also matter.


In [7]:
input_size, hidden_size = 4, 16
lstm_compare = nn.LSTM(input_size, hidden_size, batch_first=True)
gru_compare = nn.GRU(input_size, hidden_size, batch_first=True)

lstm_parameters = sum(p.numel() for p in lstm_compare.parameters())
gru_parameters = sum(p.numel() for p in gru_compare.parameters())
print("LSTM parameters:", lstm_parameters)  # 1408
print("GRU parameters:", gru_parameters)    # 1056
print("GRU reduction:", f"{100 * (1 - gru_parameters / lstm_parameters):.1f}%")


LSTM parameters: 1408
GRU parameters: 1056
GRU reduction: 25.0%


## 8. Small recurrent sentiment-classification demonstration

This is a compact application of recurrent models, not a full NLP preprocessing lesson. The tiny dataset used during learning is not adequate for conclusions about generalization. Vocabulary construction and preprocessing will be studied systematically in Chapter 10.


In [8]:
from torch.nn.utils.rnn import pack_padded_sequence

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, cell) = self.lstm(packed)
        return self.fc(hidden[-1])


class SentimentGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.gru = nn.GRU(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        return self.fc(hidden[-1])


### Input shape contract

- `input_ids`: `(batch_size, padded_sequence_length)`, dtype `torch.long`
- `lengths`: `(batch_size,)`, dtype `torch.long`, containing each sequence's true length before padding
- Output logits: `(batch_size, 2)`

`padding_idx` prevents the padding embedding row from receiving gradient updates. Packing prevents the recurrent layer from processing padding positions as if they were real tokens.


## 9. Key takeaways

- RNNs process ordered data by updating a hidden state across time steps.
- Vanilla RNNs can struggle with long-range dependencies due to vanishing or exploding gradients.
- LSTMs use a cell state and three gates to regulate information flow.
- GRUs use two gates and do not have a separate cell state.
- Fewer parameters do not guarantee proportionally faster training or worse predictive performance.
- Longer sequences increase recurrent computation and the intermediate information needed for BPTT.
- Tiny toy datasets teach implementation mechanics, not robust generalization.

## Exercises

1. Change the sequence length in the next-step prediction example. What changes in input shape and computation?
2. Compare LSTM and GRU parameter counts for several input and hidden sizes.
3. Explain why parameter-count differences do not imply the same percentage difference in wall-clock training time.
4. Explain why validation and test data must not be used to update model weights.
5. Explain why `hidden[-1]` has shape `(batch_size, hidden_size)` for a one-layer, unidirectional model.
